## C++ acceleration (pybind11)

The next two cells build the extension `ambs_sd_core` (C++ port of AMBS-SD), which runs the grid search and the 10 000-trajectory evaluation. 

In [ ]:
%%writefile ambs_sd_core.cpp
// ambs_sd_core.cpp
// C++ core of AMBS-SD for the S-CLSP-SD: the AMBS of ambs_core.cpp plus two extensions.
//  (1) Sequencing: the sequence of a period starts with the item currently set up and ends with the item
//      AMBS selects as last product. New setups are inserted by cheapest insertion (smallest increase of
//      k_sd; ties: smaller setup-time increase, then later position). k_sd[u][v] and st_sd[u][v] are
//      charged along the sequence, both when planning (capacity check) and when the plan is realized.
//  (2) Setup-cost-aware selection: for an item that needs a NEW setup, the backorder priority
//      max(EBO_i - Bmin, 0) is divided by (1 + lam * dc_i / c_bar), with dc_i its cheapest-insertion
//      setup-cost increase and c_bar the mean off-diagonal setup cost. lam is calibrated with
//      (Bmin, Hmax, Zmax).
// With sequence-independent matrices and lam = 0 it reduces exactly to ambs_core.cpp.
// SD-only safeguards: a new setup is opened only if one unit can still be produced after its setup
// time, and the item set up at the start of the next period is the one the machine is actually set up
// for after the realized sequence.
// RNG: per-trajectory std::mt19937_64, as in ambs_core.cpp (not bit-identical to numpy).

#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <vector>
#include <random>
#include <cmath>
#include <algorithm>
#include <limits>
#include <numeric>
#ifdef _OPENMP
#include <omp.h>
#endif

namespace py = pybind11;

static const double EPS = 1e-9;

// Instance
struct InstanceSD {
    int K;
    std::vector<double> mu, sigma, h, b;
    std::vector<std::vector<double>> k_sd, st_sd;   // [from][to], diag = 0
    double capacity;

    std::vector<double> k_eoq;              // mean incoming setup cost per item (EOQ only)
    std::vector<long long> Q;               // EOQ
    std::vector<double> C;                  // EOQ cost
    std::vector<long long> lower, upper;    // demand bounds
    double c_bar;                           // mean off-diagonal setup cost

    void finalize() {
        k_eoq.assign(K, 0.0); Q.resize(K); C.resize(K); lower.resize(K); upper.resize(K);
        double tot = 0.0; long long cnt = 0;
        for (int i = 0; i < K; ++i) {
            double s = 0.0;
            for (int j = 0; j < K; ++j) if (j != i) { s += k_sd[j][i]; tot += k_sd[j][i]; ++cnt; }
            k_eoq[i] = (K > 1) ? s / (double)(K - 1) : 0.0;
        }
        c_bar = (cnt > 0 && tot > 0.0) ? tot / (double)cnt : 1.0;
        for (int i = 0; i < K; ++i) {
            Q[i] = (long long)std::ceil(std::sqrt(2.0 * mu[i] * k_eoq[i] / h[i]));
            if (Q[i] < 1) Q[i] = 1;
            C[i] = k_eoq[i] * mu[i] / (double)Q[i] + h[i] * (double)Q[i] / 2.0;
            long long lo = (long long)std::llround(mu[i] - sigma[i]);
            if (lo < 0) lo = 0;
            lower[i] = lo;
            upper[i] = (long long)std::llround(mu[i] + sigma[i]);
        }
    }
};

struct AMBSParamsSD {
    double Bmin;
    double Hmax;
    int Zmax;
    double lam;
};

// Demand
static void sample_demand(const InstanceSD& inst, std::mt19937_64& rng, std::vector<long long>& d) {
    d.resize(inst.K);
    for (int i = 0; i < inst.K; ++i) {
        std::uniform_int_distribution<long long> dist(inst.lower[i], inst.upper[i]);
        d[i] = dist(rng);
    }
}

static inline double expected_backlog(const InstanceSD& inst, int i, double x) {
    long long lo = inst.lower[i], hi = inst.upper[i];
    double p = 1.0 / (double)(hi - lo + 1);
    double total = 0.0;
    for (long long dd = lo; dd <= hi; ++dd) {
        double diff = (double)dd - x;
        if (diff > 0.0) total += p * diff;
    }
    return total;
}

// Cheapest insertion of item i into the path  start -> path[0] -> ... -> path[n-1] [-> end]
// (end < 0 means open end). Returns position (0..n), cost and time increase.
// Ties: smaller time increase, then LATER position.
static void best_insertion(const InstanceSD& inst, int start, const std::vector<int>& path, int end, int i,
                           int& best_pos, double& best_dc, double& best_dt) {
    int n = (int)path.size();
    best_pos = -1; best_dc = 0.0; best_dt = 0.0;
    for (int pos = n; pos >= 0; --pos) {
        int prev = (pos == 0) ? start : path[pos - 1];
        int nxt = (pos < n) ? path[pos] : end;
        double dc = inst.k_sd[prev][i];
        double dt = inst.st_sd[prev][i];
        if (nxt >= 0) {
            dc += inst.k_sd[i][nxt] - inst.k_sd[prev][nxt];
            dt += inst.st_sd[i][nxt] - inst.st_sd[prev][nxt];
        }
        if (best_pos < 0 || dc < best_dc - EPS || (std::fabs(dc - best_dc) <= EPS && dt < best_dt - EPS)) {
            best_pos = pos; best_dc = dc; best_dt = dt;
        }
    }
}

// AMBS-SD: production plan (mirrors ambs_core.cpp::determine_production_plan)
static void determine_production_plan(const InstanceSD& inst, const std::vector<long long>& inventory,
                                       int iSU, double used_capacity, const AMBSParamsSD& params,
                                       std::vector<long long>& planned_production,
                                       std::vector<int>& new_setup_skus,   // selection order
                                       std::vector<int>& path,             // cheapest-insertion order
                                       int& last_product_scheduled) {
    const int K = inst.K;
    const double capacity = inst.capacity;
    planned_production.assign(K, 0);
    new_setup_skus.clear();
    path.clear();
    long long units = 0;
    double setup_time = 0.0;
    auto planned_capacity = [&]() { return used_capacity + (double)units + setup_time; };
    auto room_for_unit = [&]() { return planned_capacity() + 1.0 <= capacity + EPS; };

    std::vector<char> in_new(K, 0), blocked(K, 0);
    bool last_product_determined = false;
    last_product_scheduled = -1;

    std::vector<double> ebo_raw(K), score(K), ins_dc(K), ins_dt(K);
    std::vector<int> ins_pos(K);
    for (int i = 0; i < K; ++i) ebo_raw[i] = expected_backlog(inst, i, (double)inventory[i]);

    auto refresh_insertions = [&]() {
        for (int i = 0; i < K; ++i) {
            if (i == iSU || in_new[i]) { ins_dc[i] = 0.0; ins_dt[i] = 0.0; ins_pos[i] = -1; continue; }
            best_insertion(inst, iSU, path, -1, i, ins_pos[i], ins_dc[i], ins_dt[i]);
        }
    };
    refresh_insertions();

    bool stop = false;
    while (!stop && room_for_unit()) {
        const bool can_open = (int)new_setup_skus.size() < params.Zmax;
        for (int i = 0; i < K; ++i) {
            const bool is_new_setup = !(in_new[i] || i == iSU);
            const bool eligible = !is_new_setup || (can_open && !blocked[i]);
            double v = eligible ? ebo_raw[i] - params.Bmin : -params.Bmin;
            v = v > 0.0 ? v : 0.0;
            if (v > 0.0 && is_new_setup && params.lam > 0.0)
                v /= (1.0 + params.lam * ins_dc[i] / inst.c_bar);
            score[i] = v;
        }
        int i_best = (int)(std::max_element(score.begin(), score.end()) - score.begin());

        if (score[i_best] > 0.0) {
            if (!in_new[i_best] && i_best != iSU) {
                // SD safeguard: the setup must leave room for at least one unit
                if (planned_capacity() + ins_dt[i_best] + 1.0 > capacity + EPS) {
                    blocked[i_best] = 1;
                    continue;
                }
                new_setup_skus.push_back(i_best);
                in_new[i_best] = 1;
                path.insert(path.begin() + ins_pos[i_best], i_best);
                setup_time += ins_dt[i_best];
                refresh_insertions();
            }
            planned_production[i_best] += 1;
            units += 1;
            ebo_raw[i_best] = expected_backlog(inst, i_best, (double)(inventory[i_best] + planned_production[i_best]));
        } else {
            std::vector<int> produced_skus;
            produced_skus.push_back(iSU);
            for (int s : new_setup_skus) produced_skus.push_back(s);

            if ((int)produced_skus.size() <= 2) {
                last_product_determined = true;
                last_product_scheduled = (produced_skus.size() == 1) ? produced_skus[0] : produced_skus[1];
                int i0 = produced_skus[0];
                while (room_for_unit()) {
                    if ((double)(inventory[i0] + planned_production[i0] + 1) < params.Hmax * (double)inst.Q[i0]) {
                        planned_production[i0] += 1; units += 1;
                    } else break;
                }
            } else {
                std::vector<double> inv_ratio_new;
                for (int s : new_setup_skus)
                    inv_ratio_new.push_back((double)(planned_production[s] + inventory[s]) / (double)inst.Q[s]);
                int a = (int)(std::min_element(inv_ratio_new.begin(), inv_ratio_new.end()) - inv_ratio_new.begin());
                last_product_scheduled = new_setup_skus[a];
                last_product_determined = true;
                produced_skus.erase(std::remove(produced_skus.begin(), produced_skus.end(), last_product_scheduled),
                                    produced_skus.end());
                std::vector<double> inv_ratio;
                for (int s : produced_skus)
                    inv_ratio.push_back((double)(planned_production[s] + inventory[s]) / (double)inst.Q[s]);
                while (room_for_unit() && !produced_skus.empty()) {
                    int j = (int)(std::min_element(inv_ratio.begin(), inv_ratio.end()) - inv_ratio.begin());
                    int s = produced_skus[j];
                    if ((double)(inventory[s] + planned_production[s] + 1) < params.Hmax * (double)inst.Q[s]) {
                        planned_production[s] += 1; units += 1;
                    } else {
                        produced_skus.erase(produced_skus.begin() + j);
                        inv_ratio.erase(inv_ratio.begin() + j);
                    }
                }
            }
            while (!stop && room_for_unit() && last_product_determined) {
                int s = last_product_scheduled;
                if ((double)(inventory[s] + planned_production[s] + 1) < params.Hmax * (double)inst.Q[s]) {
                    planned_production[s] += 1; units += 1;
                } else {
                    stop = true;   // force stop, mirrors reference
                }
            }
            stop = true;   // the reference loop always exits after this branch
        }
    }

    if (!last_product_determined) {
        if (new_setup_skus.empty()) {
            last_product_scheduled = iSU;
        } else {
            std::vector<double> inv_ratio_new;
            for (int s : new_setup_skus)
                inv_ratio_new.push_back((double)(planned_production[s] + inventory[s]) / (double)inst.Q[s]);
            int a = (int)(std::min_element(inv_ratio_new.begin(), inv_ratio_new.end()) - inv_ratio_new.begin());
            last_product_scheduled = new_setup_skus[a];
        }
    }
}

// Final sequence: iSU (if produced and not last) -> intermediates (re-inserted
// by cheapest insertion with the last product fixed at the end) -> last.
static void build_sequence(const InstanceSD& inst, const std::vector<long long>& planned_production,
                           const std::vector<int>& path, int iSU, int last, std::vector<int>& order) {
    order.clear();
    if (planned_production[iSU] > 0 && iSU != last) order.push_back(iSU);
    const bool last_in = (planned_production[last] > 0 || last == iSU);
    const int end = (last_in && last != iSU) ? last : -1;
    std::vector<int> mid;
    for (int m : path) {
        if (m == last || m == iSU || planned_production[m] <= 0) continue;
        int pos; double dc, dt;
        best_insertion(inst, iSU, mid, end, m, pos, dc, dt);
        mid.insert(mid.begin() + pos, m);
    }
    for (int m : mid) order.push_back(m);
    if (last_in) order.push_back(last);
}

static void realize_production_plan(const InstanceSD& inst, const std::vector<long long>& planned_production,
                                     const std::vector<int>& order, int iSU, double used_capacity,
                                     std::vector<long long>& actual, int& n_setups, double& setup_cost_total,
                                     double& setup_time_total, int& final_setup) {
    const int K = inst.K;
    const double capacity = inst.capacity;
    actual.assign(K, 0);
    double capacity_used = used_capacity;
    n_setups = 0; setup_cost_total = 0.0; setup_time_total = 0.0;
    int current_setup = iSU;
    for (int i : order) {
        double remaining = capacity - capacity_used;
        if (remaining < 1.0 - EPS) break;
        if (i != current_setup) {
            double st = inst.st_sd[current_setup][i];
            if (remaining - st < 1.0 - EPS) continue;
            n_setups += 1;
            setup_cost_total += inst.k_sd[current_setup][i];
            setup_time_total += st;
            capacity_used += st;
            current_setup = i;
        }
        long long room = (long long)std::floor(capacity - capacity_used + EPS);
        long long q = std::min(planned_production[i], room);
        if (q <= 0) continue;
        actual[i] = q;
        capacity_used += (double)q;
    }
    final_setup = current_setup;
}

// One period (shared by the random and the given-demand simulators)
struct PeriodOut { double hold, bo, setup_cost, setup_time; int n_setups; long long units; };

static PeriodOut run_period(const InstanceSD& inst, const AMBSParamsSD& params,
                            std::vector<long long>& inventory, int& iSU, const std::vector<long long>& demand,
                            std::vector<int>* order_out = nullptr) {
    std::vector<long long> plan, actual;
    std::vector<int> new_setups, path, order;
    int last;
    determine_production_plan(inst, inventory, iSU, 0.0, params, plan, new_setups, path, last);
    build_sequence(inst, plan, path, iSU, last, order);
    PeriodOut o{0, 0, 0, 0, 0, 0};
    int final_setup;
    realize_production_plan(inst, plan, order, iSU, 0.0, actual, o.n_setups, o.setup_cost, o.setup_time, final_setup);
    if (order_out) *order_out = order;
    for (int i = 0; i < inst.K; ++i) { inventory[i] += actual[i] - demand[i]; o.units += actual[i]; }
    for (int i = 0; i < inst.K; ++i) {
        if (inventory[i] > 0) o.hold += inst.h[i] * (double)inventory[i];
        else o.bo += inst.b[i] * (double)(-inventory[i]);
    }
    iSU = final_setup;
    return o;
}

static std::vector<long long> initial_inventory(const InstanceSD& inst) {
    std::vector<long long> inv(inst.K);
    for (int i = 0; i < inst.K; ++i) inv[i] = (long long)std::ceil(inst.mu[i] + 1.282 * inst.sigma[i]);
    return inv;
}

struct TrajStats { double cost, hold, bo, setup_cost, setups, setup_time, units; };

static TrajStats simulate_trajectory(const InstanceSD& inst, const AMBSParamsSD& params,
                                     std::mt19937_64& rng, int warmup_periods, int periods_measured) {
    std::vector<long long> inventory = initial_inventory(inst), d;
    int iSU = 0;
    TrajStats s{0, 0, 0, 0, 0, 0, 0};
    for (int t = 0; t < warmup_periods + periods_measured; ++t) {
        sample_demand(inst, rng, d);
        PeriodOut o = run_period(inst, params, inventory, iSU, d);
        if (t >= warmup_periods) {
            s.hold += o.hold; s.bo += o.bo; s.setup_cost += o.setup_cost;
            s.setups += o.n_setups; s.setup_time += o.setup_time; s.units += (double)o.units;
        }
    }
    double n = (double)periods_measured;
    s.hold /= n; s.bo /= n; s.setup_cost /= n; s.setups /= n; s.setup_time /= n; s.units /= n;
    s.cost = s.hold + s.bo + s.setup_cost;
    return s;
}

static std::mt19937_64 make_trajectory_rng(long long base_seed, long long trajectory_index) {
    std::seed_seq seq{
        (uint32_t)(base_seed & 0xffffffffULL),
        (uint32_t)((base_seed >> 32) & 0xffffffffULL),
        (uint32_t)(trajectory_index & 0xffffffffULL),
        (uint32_t)((trajectory_index >> 32) & 0xffffffffULL)
    };
    return std::mt19937_64(seq);
}

static TrajStats evaluate_ambs(const InstanceSD& inst, const AMBSParamsSD& params, int n_trajectories,
                               long long base_seed, int warmup_periods, int periods_measured) {
    TrajStats m{0, 0, 0, 0, 0, 0, 0};
    for (int i = 0; i < n_trajectories; ++i) {
        std::mt19937_64 rng = make_trajectory_rng(base_seed, i);
        TrajStats s = simulate_trajectory(inst, params, rng, warmup_periods, periods_measured);
        m.cost += s.cost; m.hold += s.hold; m.bo += s.bo; m.setup_cost += s.setup_cost;
        m.setups += s.setups; m.setup_time += s.setup_time; m.units += s.units;
    }
    double n = (double)n_trajectories;
    m.cost /= n; m.hold /= n; m.bo /= n; m.setup_cost /= n; m.setups /= n; m.setup_time /= n; m.units /= n;
    return m;
}

// Python bindings
static InstanceSD build_instance(int K, std::vector<double> mu, std::vector<double> sigma,
                                 std::vector<double> h, std::vector<double> b,
                                 std::vector<std::vector<double>> k_sd, std::vector<std::vector<double>> st_sd,
                                 double capacity) {
    InstanceSD inst;
    inst.K = K; inst.mu = std::move(mu); inst.sigma = std::move(sigma);
    inst.h = std::move(h); inst.b = std::move(b);
    inst.k_sd = std::move(k_sd); inst.st_sd = std::move(st_sd);
    inst.capacity = capacity;
    inst.finalize();
    return inst;
}

static py::dict stats_to_dict(const TrajStats& s, double scale) {
    py::dict d;
    d["cost"] = s.cost / scale; d["holding"] = s.hold / scale; d["backorder"] = s.bo / scale;
    d["setup_cost"] = s.setup_cost / scale; d["setups"] = s.setups; d["setup_time"] = s.setup_time;
    d["units"] = s.units;
    return d;
}

py::dict py_evaluate(int K, std::vector<double> mu, std::vector<double> sigma,
                     std::vector<double> h, std::vector<double> b,
                     std::vector<std::vector<double>> k_sd, std::vector<std::vector<double>> st_sd,
                     double capacity, double Bmin, double Hmax, int Zmax, double lam,
                     int n_trajectories, long long base_seed, int warmup_periods, int periods_measured,
                     bool per_product) {
    InstanceSD inst = build_instance(K, mu, sigma, h, b, k_sd, st_sd, capacity);
    AMBSParamsSD params{Bmin, Hmax, Zmax, lam};
    TrajStats s = evaluate_ambs(inst, params, n_trajectories, base_seed, warmup_periods, periods_measured);
    return stats_to_dict(s, per_product ? (double)K : 1.0);
}

py::dict py_grid_search(int K, std::vector<double> mu, std::vector<double> sigma,
                        std::vector<double> h, std::vector<double> b,
                        std::vector<std::vector<double>> k_sd, std::vector<std::vector<double>> st_sd,
                        double capacity, std::vector<double> xb_grid, std::vector<double> xh_grid,
                        std::vector<int> z_grid, std::vector<double> lam_grid,
                        int n_trajectories, long long base_seed, int warmup_periods, int periods_measured,
                        int n_threads) {
    InstanceSD inst = build_instance(K, mu, sigma, h, b, k_sd, st_sd, capacity);
    double Bmin_scale = std::accumulate(inst.C.begin(), inst.C.end(), 0.0) / (double)K;   // mean(C)

    struct Combo { double xb, xh; int z; double lam; };
    std::vector<Combo> combos;
    for (double xb : xb_grid) for (double xh : xh_grid) for (int z : z_grid) for (double lam : lam_grid)
        combos.push_back({xb, xh, z, lam});
    int n = (int)combos.size();
    std::vector<double> results(n);

    {
        py::gil_scoped_release release;
#ifdef _OPENMP
        if (n_threads > 0) omp_set_num_threads(n_threads);
        #pragma omp parallel for schedule(dynamic)
#endif
        for (int idx = 0; idx < n; ++idx) {
            AMBSParamsSD p{combos[idx].xb * Bmin_scale, combos[idx].xh, combos[idx].z, combos[idx].lam};
            results[idx] = evaluate_ambs(inst, p, n_trajectories, base_seed, warmup_periods, periods_measured).cost;
        }
    }

    int best = (int)(std::min_element(results.begin(), results.end()) - results.begin());
    py::list all_results;
    for (int idx = 0; idx < n; ++idx)
        all_results.append(py::make_tuple(combos[idx].xb, combos[idx].xh, combos[idx].z, combos[idx].lam, results[idx]));
    py::dict out;
    out["Bmin"] = combos[best].xb * Bmin_scale;
    out["Hmax"] = combos[best].xh;
    out["Zmax"] = combos[best].z;
    out["lam"] = combos[best].lam;
    out["cost"] = results[best];
    out["all_results"] = all_results;
    return out;
}

// Deterministic simulation on a GIVEN demand matrix (T x K); used to check
// that the Python reference implementation and this core take identical
// decisions. Returns per-period costs and production sequences.
py::dict py_simulate_given_demands(int K, std::vector<double> mu, std::vector<double> sigma,
                                   std::vector<double> h, std::vector<double> b,
                                   std::vector<std::vector<double>> k_sd, std::vector<std::vector<double>> st_sd,
                                   double capacity, double Bmin, double Hmax, int Zmax, double lam,
                                   std::vector<std::vector<long long>> demands) {
    InstanceSD inst = build_instance(K, mu, sigma, h, b, k_sd, st_sd, capacity);
    AMBSParamsSD params{Bmin, Hmax, Zmax, lam};
    std::vector<long long> inventory = initial_inventory(inst);
    int iSU = 0;
    py::list costs, orders;
    for (const auto& d : demands) {
        std::vector<int> order;
        PeriodOut o = run_period(inst, params, inventory, iSU, d, &order);
        costs.append(o.hold + o.bo + o.setup_cost);
        orders.append(order);
    }
    py::dict out;
    out["costs"] = costs;
    out["orders"] = orders;
    return out;
}

PYBIND11_MODULE(ambs_sd_core, m) {
    m.doc() = "Fast C++ core for the AMBS-SD heuristic (grid search + evaluation)";
    m.def("evaluate", &py_evaluate, "Evaluate AMBS-SD for fixed (Bmin, Hmax, Zmax, lam)",
          py::arg("K"), py::arg("mu"), py::arg("sigma"), py::arg("h"), py::arg("b"),
          py::arg("k_sd"), py::arg("st_sd"), py::arg("capacity"),
          py::arg("Bmin"), py::arg("Hmax"), py::arg("Zmax"), py::arg("lam"),
          py::arg("n_trajectories"), py::arg("base_seed"), py::arg("warmup_periods"),
          py::arg("periods_measured"), py::arg("per_product") = false);
    m.def("grid_search", &py_grid_search, "Grid search over (xb, xh, z, lam), parallel over combos",
          py::arg("K"), py::arg("mu"), py::arg("sigma"), py::arg("h"), py::arg("b"),
          py::arg("k_sd"), py::arg("st_sd"), py::arg("capacity"),
          py::arg("xb_grid"), py::arg("xh_grid"), py::arg("z_grid"), py::arg("lam_grid"),
          py::arg("n_trajectories"), py::arg("base_seed"), py::arg("warmup_periods"),
          py::arg("periods_measured"), py::arg("n_threads") = 0);
    m.def("simulate_given_demands", &py_simulate_given_demands,
          "Deterministic AMBS-SD run on a given demand matrix (for testing)",
          py::arg("K"), py::arg("mu"), py::arg("sigma"), py::arg("h"), py::arg("b"),
          py::arg("k_sd"), py::arg("st_sd"), py::arg("capacity"),
          py::arg("Bmin"), py::arg("Hmax"), py::arg("Zmax"), py::arg("lam"), py::arg("demands"));
}


In [ ]:
import subprocess, sys, sysconfig, os

def _pip_install(pkg):
    """Install with pip; retry with --break-system-packages on externally-managed (PEP 668) environments."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                        capture_output=True, text=True)
    if r.returncode != 0 and "externally-managed-environment" in (r.stderr or ""):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                             "--break-system-packages", pkg],
                            capture_output=True, text=True)
    return r

pip_result = _pip_install("pybind11")
if pip_result.returncode != 0:
    print("--- pip stdout ---"); print(pip_result.stdout)
    print("--- pip stderr ---"); print(pip_result.stderr)
    raise RuntimeError("pip install pybind11 failed. See output above.")

# Build in the kernel's working directory so the .so can be imported.
build_dir = os.getcwd()
print("Building in:", build_dir)

py_include = sysconfig.get_path("include")
pybind_include = subprocess.run(
    [sys.executable, "-c", "import pybind11; print(pybind11.get_include())"],
    capture_output=True, text=True, check=True
).stdout.strip()
ext_suffix = sysconfig.get_config_var("EXT_SUFFIX")
so_path = os.path.join(build_dir, f"ambs_sd_core{ext_suffix}")

cmd = [
    "g++", "-O3", "-Wall", "-shared", "-std=c++17", "-fPIC", "-fopenmp",
    f"-I{py_include}", f"-I{pybind_include}",
    os.path.join(build_dir, "ambs_sd_core.cpp"),
    "-o", so_path,
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

print("--- compiler stdout ---")
print(result.stdout)
print("--- compiler stderr ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Compilation failed (exit code {result.returncode}). See stderr above.")

if not os.path.exists(so_path):
    raise RuntimeError(f"Compiler reported success but {so_path} was not created.")

print(f"\nBuild OK -> {so_path}")

# Make sure the current directory is importable, then load the module.
if build_dir not in sys.path:
    sys.path.insert(0, build_dir)

import importlib
if "ambs_sd_core" in sys.modules:
    importlib.reload(sys.modules["ambs_sd_core"])
import ambs_sd_core

print("ambs_sd_core module loaded OK. Detected CPU cores:", os.cpu_count())


In [ ]:
"""AMBS-SD heuristic for the S-CLSP-SD (Case II)."""

from __future__ import annotations

from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np

import ambs_sd_core  # compiled C++ extension (see the two cells above)

# 0) Global configuration
SEED = 123
INSTANCE_ID = "2.1"

# Run a quick check that the Python reference implementation and the C++ core take identical decisions on this instance before the grid search.
RUN_CONSISTENCY_CHECK = True


# 1) Instance registry (paper ID -> builder parameters)
CASE_II_REGISTRY = {
    # Type A: two families
    "2.1":  dict(sd_type="A", K=15, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.2":  dict(sd_type="A", K=15, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.3":  dict(sd_type="A", K=15, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.4":  dict(sd_type="A", K=15, cf=1.5, cov="high"),  # D_i = U{0,8}
    "2.5":  dict(sd_type="A", K=20, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.6":  dict(sd_type="A", K=20, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.7":  dict(sd_type="A", K=20, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.8":  dict(sd_type="A", K=20, cf=1.5, cov="high"),  # D_i = U{0,8}

    # Type B: five families (cost/time grow with the family distance)
    "2.9":  dict(sd_type="B", K=15, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.10": dict(sd_type="B", K=15, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.11": dict(sd_type="B", K=15, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.12": dict(sd_type="B", K=15, cf=1.5, cov="high"),  # D_i = U{0,8}
    "2.13": dict(sd_type="B", K=20, cf=1.1, cov="low"),   # D_i = U{3,5}
    "2.14": dict(sd_type="B", K=20, cf=1.5, cov="low"),   # D_i = U{3,5}
    "2.15": dict(sd_type="B", K=20, cf=1.1, cov="high"),  # D_i = U{0,8}
    "2.16": dict(sd_type="B", K=20, cf=1.5, cov="high"),  # D_i = U{0,8}
}

# Type A: two families split by index parity (even indices vs odd indices)

# Type A setup parameters
FAM_SAME_SETUP_TIME, FAM_SAME_SETUP_COST = 0.0, 100.0
FAM_DIFF_SETUP_TIME, FAM_DIFF_SETUP_COST = 1.0, 200.0

# Type B: five families on a line, family(i) = i mod 5, D = |f_i - f_j|; see module docstring.
FAM5_N_FAMILIES = 5
FAM5_SAME_SETUP_TIME, FAM5_SAME_SETUP_COST = 0.0, 100.0
FAM5_SETUP_TIME_PER_STEP = 0.5
FAM5_SETUP_COST_BASE, FAM5_SETUP_COST_PER_STEP = 100.0, 100.0


# 2) Instance model
@dataclass
class CLSPInstanceSD:
    """No-batch-size S-CLSP-SD instance.

    :param K: Number of products.
    :param mu: Mean demand per product.
    :param sigma: Half-range of the discrete-uniform demand (DynaPlex convention).
    :param h: Holding cost per product.
    :param b: Backorder cost per product.
    :param k_sd: Setup-cost matrix, k_sd[u, v] = cost of switching from u to v (diag 0).
    :param st_sd: Setup-time matrix, st_sd[u, v] = capacity used switching from u to v (diag 0).
    :param capacity: Per-period capacity (raw units).
    """
    K: int
    mu: np.ndarray
    sigma: np.ndarray
    h: np.ndarray
    b: np.ndarray
    k_sd: np.ndarray
    st_sd: np.ndarray
    capacity: int


# Target mean of the off-diagonal setup costs (same level as the Case I setup cost k = 200)
SETUP_COST_MEAN = 200.0


def normalize_setup_cost(k_sd, target=SETUP_COST_MEAN):
    """Rescale a setup-cost matrix so that the mean of its off-diagonal entries equals `target`."""
    off = ~np.eye(k_sd.shape[0], dtype=bool)
    return k_sd * (target / k_sd[off].mean())


def build_setup_matrices_families(K: int) -> Tuple[np.ndarray, np.ndarray]:
    """Type A setups: two families by index parity (see module docstring). Costs are rescaled in
    build_inst_case2.
    """
    fam = np.arange(K) % 2
    same = fam[:, None] == fam[None, :]
    st = np.where(same, FAM_SAME_SETUP_TIME, FAM_DIFF_SETUP_TIME).astype(float)
    k = np.where(same, FAM_SAME_SETUP_COST, FAM_DIFF_SETUP_COST).astype(float)
    np.fill_diagonal(st, 0.0)
    np.fill_diagonal(k, 0.0)
    return k, st


def build_setup_matrices_families5(K: int) -> Tuple[np.ndarray, np.ndarray]:
    """Type B setups: five families on a line (see module docstring). Diagonal 0; costs are rescaled by
    normalize_setup_cost.
    """
    fam = np.arange(K) % FAM5_N_FAMILIES
    D = np.abs(fam[:, None] - fam[None, :]).astype(float)
    same = D == 0
    st = np.where(same, FAM5_SAME_SETUP_TIME, FAM5_SETUP_TIME_PER_STEP * D)
    k = np.where(same, FAM5_SAME_SETUP_COST, FAM5_SETUP_COST_BASE + FAM5_SETUP_COST_PER_STEP * D)
    np.fill_diagonal(st, 0.0)
    np.fill_diagonal(k, 0.0)
    return k, st


def build_inst_case2(sd_type: str, K: int, cov_level: str, cf: float,
                     mu_value: float = 4.0, h_value: float = 1.0, b_value: float = 9.0) -> CLSPInstanceSD:
    """Case II instance (see module docstring). Capacity = int(K * mu * cf), as in Case I."""
    assert sd_type in ("A", "B") and cov_level in ("low", "high")
    sigma_value = 1.0 if cov_level == "low" else mu_value
    if sd_type == "A":
        k_sd, st_sd = build_setup_matrices_families(K)
    else:
        k_sd, st_sd = build_setup_matrices_families5(K)
    k_sd = normalize_setup_cost(k_sd)               # mean off-diagonal setup cost = 200
    return CLSPInstanceSD(
        K=K,
        mu=np.full(K, mu_value, dtype=float),
        sigma=np.full(K, sigma_value, dtype=float),
        h=np.full(K, h_value, dtype=float),
        b=np.full(K, b_value, dtype=float),
        k_sd=k_sd, st_sd=st_sd,
        capacity=int(K * mu_value * cf),
    )


def build_inst_from_instance_id(instance_id: str) -> Tuple[CLSPInstanceSD, str]:
    instance_id = instance_id.strip()
    if instance_id not in CASE_II_REGISTRY:
        raise ValueError(f"Unknown INSTANCE_ID '{instance_id}' for S-CLSP-SD (Case II IDs are 2.1..2.16).")
    p = CASE_II_REGISTRY[instance_id]
    inst = build_inst_case2(p["sd_type"], p["K"], p["cov"], p["cf"])
    exp_name = f"ID{instance_id}_type{p['sd_type']}_K{p['K']}_cov{p['cov']}_cf{p['cf']}"
    return inst, exp_name


# 3) Demand model - discrete uniform U[round(mu-sigma), round(mu+sigma)]
class DemandGenerator:
    def __init__(self, inst: CLSPInstanceSD):
        self.lower = np.maximum(np.round(inst.mu - inst.sigma).astype(np.int64), 0)
        self.upper = np.round(inst.mu + inst.sigma).astype(np.int64)

    def sample(self, rng: np.random.Generator) -> np.ndarray:
        return rng.integers(self.lower, self.upper + 1, dtype=np.int64)

    def expected_backlog(self, i: int, x: float) -> float:
        """E[max(d_i - x, 0)] in UNITS (not multiplied by b_i), as in the original AMBS."""
        lo, hi = int(self.lower[i]), int(self.upper[i])
        p = 1.0 / (hi - lo + 1)
        total = 0.0
        for d in range(lo, hi + 1):
            diff = d - x
            if diff > 0.0:
                total += p * diff
        return total


# 4) EOQ helpers (mean incoming setup cost as k_i)
def eoq_setup_cost(inst: CLSPInstanceSD) -> Tuple[np.ndarray, float]:
    """k_i = mean_{j != i} k_sd[j, i] (mean cost of setting the machine up for i), and c_bar = mean
    off-diagonal setup cost. Plain sequential sums, in the same order as ambs_sd_core.cpp, so both give
    bit-identical values.
    """
    K = inst.K
    k_eoq = np.zeros(K, dtype=float)
    tot, cnt = 0.0, 0
    for i in range(K):
        s = 0.0
        for j in range(K):
            if j != i:
                s += float(inst.k_sd[j, i])
                tot += float(inst.k_sd[j, i])
                cnt += 1
        k_eoq[i] = s / (K - 1) if K > 1 else 0.0
    c_bar = tot / cnt if (cnt > 0 and tot > 0.0) else 1.0
    return k_eoq, c_bar


def eoq(mu, k, h) -> np.ndarray:
    return np.maximum(np.ceil(np.sqrt(2.0 * mu * k / h)), 1).astype(np.int64)


def eoq_cost(mu, k, h, Q) -> np.ndarray:
    return k * mu / Q + h * Q / 2.0


# 5) Parameters / results
@dataclass
class AMBSParamsSD:
    Bmin: float
    Hmax: float
    Zmax: int
    lam: float = 0.0


@dataclass
class AMBSResultSD:
    avg_cost: float
    avg_inv_cost: float
    avg_bo_cost: float
    avg_setup_cost: float
    avg_setups: float
    avg_setup_time: float
    avg_lot_size_units: float


def initial_inventory(mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    """ceil(mu + 1.282*sigma) -- base-stock 90%, deterministic (machine set up for item 0)."""
    return np.ceil(mu + 1.282 * sigma).astype(np.int64)


# 6) AMBS-SD heuristic (Python reference; mirrors ambs_sd_core.cpp line by line)
EPS = 1e-9


class AMBSSDHeuristic:
    def __init__(self, inst: CLSPInstanceSD):
        self.inst = inst
        self.demand = DemandGenerator(inst)
        self.k_eoq, self.c_bar = eoq_setup_cost(inst)
        self.Q = eoq(inst.mu, self.k_eoq, inst.h)
        self.C = eoq_cost(inst.mu, self.k_eoq, inst.h, self.Q)

    # sequencing
    def best_insertion(self, start: int, path: List[int], end: int, i: int) -> Tuple[int, float, float]:
        """Cheapest insertion of i into start -> path -> [end] (end < 0: open end). Returns (position, cost
        increase, time increase); ties -> smaller time, then later position.
        """
        k, st = self.inst.k_sd, self.inst.st_sd
        n = len(path)
        best = None
        for pos in range(n, -1, -1):
            prev = start if pos == 0 else path[pos - 1]
            nxt = path[pos] if pos < n else end
            dc = k[prev, i]
            dt = st[prev, i]
            if nxt >= 0:
                dc += k[i, nxt] - k[prev, nxt]
                dt += st[i, nxt] - st[prev, nxt]
            if best is None or dc < best[1] - EPS or (abs(dc - best[1]) <= EPS and dt < best[2] - EPS):
                best = (pos, float(dc), float(dt))
        return best

    def determine_production_plan(self, inventory: np.ndarray, iSU: int, params: AMBSParamsSD,
                                  used_capacity: float = 0.0):
        """AMBS plan with SD setup-time accounting (cheapest insertion) and the lam penalty. Returns
        (planned_production, new_setup_skus, path, last_product_scheduled).
        """
        inst = self.inst
        K, capacity = inst.K, float(inst.capacity)
        planned = np.zeros(K, dtype=np.int64)
        new_setup_skus: List[int] = []
        path: List[int] = []
        units, setup_time = 0, 0.0
        in_new = np.zeros(K, dtype=bool)
        blocked = np.zeros(K, dtype=bool)
        last_determined, last = False, None

        def planned_capacity():
            return used_capacity + units + setup_time

        def room_for_unit():
            return planned_capacity() + 1.0 <= capacity + EPS

        ebo_raw = np.array([self.demand.expected_backlog(i, float(inventory[i])) for i in range(K)])
        ins = [None] * K

        def refresh_insertions():
            for i in range(K):
                ins[i] = None if (i == iSU or in_new[i]) else self.best_insertion(iSU, path, -1, i)

        refresh_insertions()
        stop = False
        while not stop and room_for_unit():
            can_open = len(new_setup_skus) < params.Zmax
            score = np.zeros(K, dtype=np.float64)
            for i in range(K):
                is_new_setup = not (in_new[i] or i == iSU)
                eligible = (not is_new_setup) or (can_open and not blocked[i])
                v = (ebo_raw[i] - params.Bmin) if eligible else -params.Bmin
                v = v if v > 0.0 else 0.0
                if v > 0.0 and is_new_setup and params.lam > 0.0:
                    v /= (1.0 + params.lam * ins[i][1] / self.c_bar)
                score[i] = v
            i_best = int(np.argmax(score))

            if score[i_best] > 0.0:
                if not in_new[i_best] and i_best != iSU:
                    pos, _dc, dt = ins[i_best]
                    if planned_capacity() + dt + 1.0 > capacity + EPS:   # SD safeguard
                        blocked[i_best] = True
                        continue
                    new_setup_skus.append(i_best)
                    in_new[i_best] = True
                    path.insert(pos, i_best)
                    setup_time += dt
                    refresh_insertions()
                planned[i_best] += 1
                units += 1
                ebo_raw[i_best] = self.demand.expected_backlog(i_best, float(inventory[i_best] + planned[i_best]))
            else:
                produced = [iSU] + new_setup_skus
                if len(produced) <= 2:
                    last_determined = True
                    last = produced[0] if len(produced) == 1 else produced[1]
                    i0 = produced[0]
                    while room_for_unit():
                        if inventory[i0] + planned[i0] + 1 < params.Hmax * self.Q[i0]:
                            planned[i0] += 1
                            units += 1
                        else:
                            break
                else:
                    ratio_new = [(planned[s] + inventory[s]) / self.Q[s] for s in new_setup_skus]
                    last = new_setup_skus[int(np.argmin(ratio_new))]
                    last_determined = True
                    produced.remove(last)
                    ratio = [(planned[s] + inventory[s]) / self.Q[s] for s in produced]
                    while room_for_unit() and len(produced) > 0:
                        j = int(np.argmin(ratio))
                        s = produced[j]
                        if inventory[s] + planned[s] + 1 < params.Hmax * self.Q[s]:
                            planned[s] += 1
                            units += 1
                        else:
                            produced.pop(j)
                            ratio.pop(j)
                while not stop and room_for_unit() and last_determined:
                    s = last
                    if inventory[s] + planned[s] + 1 < params.Hmax * self.Q[s]:
                        planned[s] += 1
                        units += 1
                    else:
                        stop = True
                stop = True

        if not last_determined:
            if len(new_setup_skus) == 0:
                last = iSU
            else:
                ratio_new = [(planned[s] + inventory[s]) / self.Q[s] for s in new_setup_skus]
                last = new_setup_skus[int(np.argmin(ratio_new))]
        return planned, new_setup_skus, path, int(last)

    def build_sequence(self, planned: np.ndarray, path: List[int], iSU: int, last: int) -> List[int]:
        """iSU (if produced and not last) -> intermediates (cheapest insertion, last fixed) -> last."""
        order = []
        if planned[iSU] > 0 and iSU != last:
            order.append(iSU)
        last_in = planned[last] > 0 or last == iSU
        end = last if (last_in and last != iSU) else -1
        mid: List[int] = []
        for m in path:
            if m == last or m == iSU or planned[m] <= 0:
                continue
            pos, _dc, _dt = self.best_insertion(iSU, mid, end, m)
            mid.insert(pos, m)
        order.extend(mid)
        if last_in:
            order.append(last)
        return order

    def realize_production_plan(self, planned: np.ndarray, order: List[int], iSU: int,
                                used_capacity: float = 0.0):
        """Produce along `order`, charging k_sd/st_sd per transition. Returns (actual, n_setups, setup_cost,
        setup_time, final_setup).
        """
        inst = self.inst
        capacity = float(inst.capacity)
        actual = np.zeros(inst.K, dtype=np.int64)
        used = float(used_capacity)
        n_setups, setup_cost, setup_time = 0, 0.0, 0.0
        current = iSU
        for i in order:
            remaining = capacity - used
            if remaining < 1.0 - EPS:
                break
            if i != current:
                st = float(inst.st_sd[current, i])
                if remaining - st < 1.0 - EPS:
                    continue
                n_setups += 1
                setup_cost += float(inst.k_sd[current, i])
                setup_time += st
                used += st
                current = i
            room = int(np.floor(capacity - used + EPS))
            q = min(int(planned[i]), room)
            if q <= 0:
                continue
            actual[i] = q
            used += q
        return actual, n_setups, setup_cost, setup_time, current

    def run_period(self, inventory: np.ndarray, iSU: int, params: AMBSParamsSD, demand: np.ndarray):
        planned, _new, path, last = self.determine_production_plan(inventory, iSU, params)
        order = self.build_sequence(planned, path, iSU, last)
        actual, n_setups, setup_cost, setup_time, final_setup = self.realize_production_plan(planned, order, iSU)
        inventory = inventory + actual - demand
        hold = float(np.sum(self.inst.h * np.maximum(inventory, 0)))
        bo = float(np.sum(self.inst.b * np.maximum(-inventory, 0)))
        info = dict(hold=hold, bo=bo, setup_cost=setup_cost, setup_time=setup_time,
                    n_setups=n_setups, units=int(actual.sum()), order=order)
        return inventory, int(final_setup), info


# 7) Trajectories (Python)
def get_trajectory_rng(base_seed: int, trajectory_index: int) -> np.random.Generator:
    return np.random.default_rng(np.random.SeedSequence(base_seed, spawn_key=(trajectory_index,)))


def simulate_trajectory(inst: CLSPInstanceSD, params: AMBSParamsSD, rng: np.random.Generator,
                        warmup_periods: int = 128, periods_measured: int = 100) -> AMBSResultSD:
    """128 warm-up + 100 measured periods; averages over the measured periods."""
    heur = AMBSSDHeuristic(inst)
    inventory = initial_inventory(inst.mu, inst.sigma)
    iSU = 0
    acc = dict(hold=0.0, bo=0.0, setup_cost=0.0, setup_time=0.0, n_setups=0.0, units=0.0)
    for t in range(warmup_periods + periods_measured):
        demand = heur.demand.sample(rng)
        inventory, iSU, info = heur.run_period(inventory, iSU, params, demand)
        if t >= warmup_periods:
            for key in acc:
                acc[key] += info[key]
    n = float(periods_measured)
    return AMBSResultSD(
        avg_cost=(acc["hold"] + acc["bo"] + acc["setup_cost"]) / n,
        avg_inv_cost=acc["hold"] / n,
        avg_bo_cost=acc["bo"] / n,
        avg_setup_cost=acc["setup_cost"] / n,
        avg_setups=acc["n_setups"] / n,
        avg_setup_time=acc["setup_time"] / n,
        avg_lot_size_units=acc["units"] / n,
    )


# 8) C++-backed evaluation and grid search
def _inst_to_cpp_args(inst: CLSPInstanceSD) -> dict:
    return dict(
        K=inst.K, mu=inst.mu.tolist(), sigma=inst.sigma.tolist(), h=inst.h.tolist(), b=inst.b.tolist(),
        k_sd=inst.k_sd.astype(float).tolist(), st_sd=inst.st_sd.astype(float).tolist(),
        capacity=float(inst.capacity),
    )


def evaluate_ambs_sd(inst: CLSPInstanceSD, params: AMBSParamsSD, n_trajectories: int = 10_000,
                     base_seed: int = 13_021_984, warmup_periods: int = 128, periods_measured: int = 100,
                     per_product: bool = False) -> dict:
    """Mean over `n_trajectories` (C++). Returns dict with cost, holding, backorder, setup_cost, setups,
    setup_time, units (per period).
    """
    return ambs_sd_core.evaluate(
        **_inst_to_cpp_args(inst), Bmin=params.Bmin, Hmax=params.Hmax, Zmax=params.Zmax, lam=params.lam,
        n_trajectories=n_trajectories, base_seed=base_seed, warmup_periods=warmup_periods,
        periods_measured=periods_measured, per_product=per_product,
    )


def grid_search_ambs_sd(inst: CLSPInstanceSD, xb_grid, xh_grid, z_grid, lam_grid,
                        n_trajectories: int = 1000, base_seed: int = 10_000,
                        warmup_periods: int = 128, periods_measured: int = 100,
                        n_threads: int = 0) -> Tuple[AMBSParamsSD, float, list]:
    """Grid search over (Bmin = xb * mean(EOQCost), Hmax = xh, Zmax = z, lam), parallel in C++."""
    res = ambs_sd_core.grid_search(
        **_inst_to_cpp_args(inst), xb_grid=list(xb_grid), xh_grid=list(xh_grid), z_grid=list(z_grid),
        lam_grid=list(lam_grid), n_trajectories=n_trajectories, base_seed=base_seed,
        warmup_periods=warmup_periods, periods_measured=periods_measured, n_threads=n_threads,
    )
    best = AMBSParamsSD(Bmin=res["Bmin"], Hmax=res["Hmax"], Zmax=res["Zmax"], lam=res["lam"])
    return best, res["cost"], res["all_results"]


def check_python_vs_cpp(inst: CLSPInstanceSD, params_list, n_periods: int = 300, seed: int = 7) -> None:
    """Run the Python reference and the C++ core on the SAME demand matrix and assert identical per-period
    costs and production sequences.
    """
    heur = AMBSSDHeuristic(inst)
    rng = np.random.default_rng(seed)
    demands = np.stack([heur.demand.sample(rng) for _ in range(n_periods)])
    for p in params_list:
        out = ambs_sd_core.simulate_given_demands(
            **_inst_to_cpp_args(inst), Bmin=p.Bmin, Hmax=p.Hmax, Zmax=p.Zmax, lam=p.lam,
            demands=demands.tolist(),
        )
        inventory, iSU = initial_inventory(inst.mu, inst.sigma), 0
        for t in range(n_periods):
            inventory, iSU, info = heur.run_period(inventory, iSU, p, demands[t])
            c_py = info["hold"] + info["bo"] + info["setup_cost"]
            assert abs(c_py - out["costs"][t]) < 1e-6, f"cost mismatch at t={t}: {c_py} vs {out['costs'][t]} ({p})"
            assert list(info["order"]) == list(out["orders"][t]), f"sequence mismatch at t={t} ({p})"
    print(f"[check] Python reference == C++ core on {n_periods} periods x {len(params_list)} parameter sets.")


# 9) Instance construction according to INSTANCE_ID
inst, exp_name = build_inst_from_instance_id(INSTANCE_ID)
_heur_tmp = AMBSSDHeuristic(inst)
_bmin_scale = float(np.mean(_heur_tmp.C))
print(f"Instance {exp_name}: capacity={inst.capacity}, EOQ={_heur_tmp.Q.tolist()}, "
      f"mean EOQCost={_bmin_scale:.3f}, mean setup cost={_heur_tmp.c_bar:.3f}")

if RUN_CONSISTENCY_CHECK:
    check_python_vs_cpp(inst, [
        AMBSParamsSD(Bmin=0.0, Hmax=1.0, Zmax=3, lam=0.0),
        AMBSParamsSD(Bmin=0.2 * _bmin_scale, Hmax=0.8, Zmax=5, lam=1.0),
        AMBSParamsSD(Bmin=0.05 * _bmin_scale, Hmax=1.0, Zmax=inst.K - 1, lam=0.5),
    ])

# 10) Execution: (i) threshold calibration; (ii) comparable evaluation
GRID_SEARCH_N_TRAJECTORIES = 1000
GRID_SEARCH_SEED_EVAL = 10_000
GRID_SEARCH_WARMUP_PERIODS = 128
GRID_SEARCH_PERIODS_MEASURED = 100

XB_GRID = [i / 10.0 for i in range(0, 11)]
XH_GRID = [i / 10.0 for i in range(5, 11)]
# Zmax is capped at 8: calibrated Case I values never exceed 5 and AMBS-SD opens ~2-3 setups per period.
Z_GRID = list(range(1, min(inst.K, 9)))
# lam = 0 switches the setup-cost penalty off, so the grid never does worse than sequencing alone.
LAM_GRID = [0.0, 1.0, 2.0, 4.0, 8.0, 16.0]

best_params, best_grid_cost, grid_results = grid_search_ambs_sd(
    inst, XB_GRID, XH_GRID, Z_GRID, LAM_GRID,
    n_trajectories=GRID_SEARCH_N_TRAJECTORIES,
    base_seed=GRID_SEARCH_SEED_EVAL,
    warmup_periods=GRID_SEARCH_WARMUP_PERIODS,
    periods_measured=GRID_SEARCH_PERIODS_MEASURED,
    n_threads=0,  # 0 = use all cores the runtime reports
)
best_lam0 = min((r for r in grid_results if r[3] == 0.0), key=lambda r: r[4])

print("\n==============================================")
print(f"AMBS-SD experiment: {exp_name}")
print("== Best thresholds found (grid) ==")
print(f"Bmin = {best_params.Bmin:.4f}, Hmax = {best_params.Hmax:.4f}, "
      f"Zmax = {best_params.Zmax}, lam = {best_params.lam:.2f}")
print(f"Cost (grid evaluation) = {best_grid_cost:.3f}")
print(f"Best with lam = 0 (sequencing only): xb={best_lam0[0]:.1f}, xh={best_lam0[1]:.1f}, "
      f"Zmax={best_lam0[2]}, cost={best_lam0[4]:.3f}")
print("==============================================\n")

# Individual runs (Python reference, detailed breakdown).
N_RUNS = 10
WARMUP_PERIODS = 128
PERIODS_MEASURED = 100
SEED_BASE_TEST = 12

runs = []
print("== Individual AMBS-SD results (SeedSequence-independent trajectories) ==")
for r in range(N_RUNS):
    res_r = simulate_trajectory(inst, best_params, get_trajectory_rng(SEED_BASE_TEST, r),
                                warmup_periods=WARMUP_PERIODS, periods_measured=PERIODS_MEASURED)
    runs.append(res_r)
    print(f"\n-- Run {r + 1}/{N_RUNS} --")
    print(f"Average total cost : {res_r.avg_cost:.3f}")
    print(f" - Inventory : {res_r.avg_inv_cost:.3f}")
    print(f" - Backorder : {res_r.avg_bo_cost:.3f}")
    print(f" - Setups    : {res_r.avg_setup_cost:.3f}")
    print(f"Setups / period : {res_r.avg_setups:.3f}")
    print(f"Setup time / period : {res_r.avg_setup_time:.3f}")
    print(f"Avg units / period : {res_r.avg_lot_size_units:.3f}")

print(f"\n== Average over {N_RUNS} runs (warmup={WARMUP_PERIODS}, measured={PERIODS_MEASURED}) ==")
print(f"Average total cost : {np.mean([x.avg_cost for x in runs]):.3f}")
print(f" - Inventory : {np.mean([x.avg_inv_cost for x in runs]):.3f}")
print(f" - Backorder : {np.mean([x.avg_bo_cost for x in runs]):.3f}")
print(f" - Setups    : {np.mean([x.avg_setup_cost for x in runs]):.3f}")
print(f"Setups / period : {np.mean([x.avg_setups for x in runs]):.3f}")
print(f"Setup time / period : {np.mean([x.avg_setup_time for x in runs]):.3f}")
print(f"Avg units / period : {np.mean([x.avg_lot_size_units for x in runs]):.3f}")
print("==============================================")

# Final evaluation (C++): 10,000 trajectories.
N_TRAJECTORIES = 10_000
final_raw = evaluate_ambs_sd(inst, best_params, n_trajectories=N_TRAJECTORIES, base_seed=SEED_BASE_TEST,
                             warmup_periods=WARMUP_PERIODS, periods_measured=PERIODS_MEASURED)
final_pp = evaluate_ambs_sd(inst, best_params, n_trajectories=N_TRAJECTORIES, base_seed=SEED_BASE_TEST,
                            warmup_periods=WARMUP_PERIODS, periods_measured=PERIODS_MEASURED, per_product=True)

print("\n==============================================")
print(f"AMBS-SD experiment: {exp_name}  ({N_TRAJECTORIES} trajectories, "
      f"{WARMUP_PERIODS}+{PERIODS_MEASURED} periods each)")
print(f"  raw (sum over products)  : mean={final_raw['cost']:.3f}  "
      f"(holding={final_raw['holding']:.3f}, backorder={final_raw['backorder']:.3f}, "
      f"setup={final_raw['setup_cost']:.3f}, setups/period={final_raw['setups']:.3f})")
print("==============================================")
